# Week 3: YOLOv8 알약 객체 탐지 및 OpenCV 시각화
데이터 다운로드 → 학습 → 평가 → OpenCV 시각화 → 결과 압축을 순서대로 수행합니다.

In [ ]:
!pip -q install 'ultralytics>=8.3,<9' 'opencv-python-headless>=4.10,<5'
import json, random, shutil
from pathlib import Path
import cv2, matplotlib.pyplot as plt, torch, ultralytics
from ultralytics import YOLO
SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
ROOT = Path('/content/week3')
OUTPUT_DIR, RUNS_DIR = ROOT / 'outputs', ROOT / 'runs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. 설정
Medical Pills는 115장(학습 92, 검증 23)의 단일 `pill` 클래스 데이터셋입니다. 처음 학습할 때 자동 다운로드됩니다.

In [ ]:
MODEL_NAME, DATA_YAML = 'yolov8n.pt', 'medical-pills.yaml'
EPOCHS, IMAGE_SIZE, BATCH_SIZE = 30, 640, 16
print({'model': MODEL_NAME, 'data': DATA_YAML, 'epochs': EPOCHS, 'imgsz': IMAGE_SIZE, 'batch': BATCH_SIZE, 'seed': SEED})

## 2. YOLOv8n 전이학습
COCO 사전학습 가중치에서 시작해 `pill` 클래스에 맞게 미세조정합니다.

In [ ]:
model = YOLO(MODEL_NAME)
model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE, seed=SEED, deterministic=True, project=str(RUNS_DIR), name='medical_pills_yolov8n', exist_ok=True, plots=True)
RUN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHT = RUN_DIR / 'weights' / 'best.pt'
print('best weight:', BEST_WEIGHT, BEST_WEIGHT.exists())

## 3. 검증 지표 저장
검증 세트에서 Precision, Recall, mAP50, mAP50-95를 계산합니다.

In [ ]:
best_model = YOLO(str(BEST_WEIGHT))
validation = best_model.val(data=DATA_YAML, imgsz=IMAGE_SIZE, split='val', project=str(RUNS_DIR), name='validation', exist_ok=True, plots=True)
metrics = {'precision': round(float(validation.box.mp), 6), 'recall': round(float(validation.box.mr), 6), 'mAP50': round(float(validation.box.map50), 6), 'mAP50-95': round(float(validation.box.map), 6), 'model': MODEL_NAME, 'epochs': EPOCHS, 'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE, 'seed': SEED, 'ultralytics_version': ultralytics.__version__}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(metrics, ensure_ascii=False, indent=2))
for filename in ['results.png', 'PR_curve.png', 'confusion_matrix.png']:
    source = RUN_DIR / filename
    if source.exists(): shutil.copy2(source, OUTPUT_DIR / filename)

## 4. OpenCV 시각화
`result.plot()` 대신 좌표와 신뢰도를 꺼내 OpenCV 함수로 직접 그립니다.

In [ ]:
def find_validation_images():
    candidates = [Path('/content/datasets/medical-pills/images/val'), Path('/content/medical-pills/images/val'), Path.home() / 'datasets/medical-pills/images/val']
    for directory in candidates:
        images = sorted([*directory.glob('*.jpg'), *directory.glob('*.jpeg'), *directory.glob('*.png')])
        if images: return images
    return [p for p in sorted(Path('/content').glob('**/medical-pills/images/val/*')) if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]

def draw_detections(image_path, result, output_path):
    image = cv2.imread(str(image_path))
    if image is None: raise FileNotFoundError(image_path)
    count = 0
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().tolist())
        conf, cls = float(box.conf[0].cpu()), int(box.cls[0].cpu())
        label = f'{result.names[cls]} {conf:.2f}'
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 200, 0), 2)
        cv2.putText(image, label, (x1, max(24, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 200, 0), 2, cv2.LINE_AA)
        count += 1
    cv2.putText(image, f'Detected pills: {count}', (16, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (30, 30, 255), 2, cv2.LINE_AA)
    cv2.imwrite(str(output_path), image)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB), count

validation_images = find_validation_images()
if len(validation_images) < 3: raise RuntimeError(f'검증 이미지를 찾지 못했습니다: {len(validation_images)}장')
sample_images = random.Random(SEED).sample(validation_images, 3)
predictions = best_model.predict(sample_images, imgsz=IMAGE_SIZE, conf=0.25, verbose=False)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for index, (image_path, prediction) in enumerate(zip(sample_images, predictions), start=1):
    rgb, count = draw_detections(image_path, prediction, OUTPUT_DIR / f'prediction_{index:02d}.jpg')
    axes[index - 1].imshow(rgb); axes[index - 1].set_title(f'Prediction {index} - {count} pills'); axes[index - 1].axis('off')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'predictions_grid.png', dpi=150, bbox_inches='tight'); plt.show()

## 5. 결과 압축 및 다운로드

In [ ]:
shutil.copy2(BEST_WEIGHT, OUTPUT_DIR / 'best.pt')
archive_path = shutil.make_archive('/content/week3_artifacts', 'zip', ROOT, 'outputs')
for path in sorted(OUTPUT_DIR.iterdir()): print('-', path.name, f'({path.stat().st_size / 1024:.1f} KB)')
from google.colab import files
files.download(archive_path)